# 10.5 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/05-mlflow/exercises.ipynb)

Run the lesson first: these exercises use its experiment.

In [ ]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

In [ ]:
import os, logging
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import mlflow, mlflow.sklearn
logging.getLogger("mlflow").setLevel(logging.ERROR)     # hide MLflow's informational notices

(ROOT / "mlruns").mkdir(exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{(ROOT / 'mlruns' / 'mlflow.db').as_posix()}")
EXPERIMENT = "shopwave-churn"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=(ROOT / "mlruns" / "artifacts").as_uri())
mlflow.set_experiment(EXPERIMENT)
print("Tracking runs in mlruns/mlflow.db at the course root")

In [ ]:
from sklearn.model_selection import train_test_split
cust = cm.build_customer_table(cm.load_sales())
X_train, X_test, y_train, y_test = train_test_split(
    cust[cm.FEATURES], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, train {len(X_train):,} / test {len(X_test):,}")

## Exercise 1: log a fifth model

Log a `KNeighborsClassifier(n_neighbors=51)` in a scaled pipeline, with the same metrics as the lesson. Where does it land on the leaderboard?

In [ ]:
# your code here

## Exercise 2: log a plot

For the logistic regression, make the ROC curve (03.7) and log it with `mlflow.log_figure(fig, "roc.png")` in a new run. Find it in the UI under the run's artifacts.

In [ ]:
# your code here

## Exercise 3: promote a challenger

Register the random forest as a second version of `shopwave-churn` and give it the alias `challenger`. Write a function that loads `@champion` and `@challenger` and prints both test AUCs. What evidence would you want before moving `@champion` to it?

In [ ]:
# your code here